In [4]:
import pandas as pd
from multicall import Call
import numpy as np
from web3 import Web3


from mainnet_launch.database.schema.full import (
    DestinationTokenValues,
    TokenValues,
    Autopools,
    DestinationStates,
    DestinationTokens,
    Destinations,
    AutopoolDestinationStates,
    Tokens,
    AutopoolStates,
)
import plotly.express as px


from mainnet_launch.abis import STATS_CALCULATOR_REGISTRY_ABI
from mainnet_launch.data_fetching.get_events import fetch_events


from mainnet_launch.database.schema.postgres_operations import (
    get_full_table_as_orm,
    get_full_table_as_df,
    insert_avoid_conflicts,
    get_subset_not_already_in_column,
    natural_left_right_using_where,
)
from mainnet_launch.data_fetching.get_state_by_block import (
    get_raw_state_by_blocks,
    safe_normalize_with_bool_success,
    build_blocks_to_use,
    identity_with_bool_success,
    get_state_by_one_block,
)
from mainnet_launch.constants import (
    ALL_CHAINS,
    ROOT_PRICE_ORACLE,
    ChainData,
)

from mainnet_launch.pages.autopool_diagnostics.lens_contract import (
    fetch_pools_and_destinations_df,
)


def _fetch_autopool_state_df(autopools: list[Autopools], missing_blocks: list[int], chain: ChainData) -> pd.DataFrame:

    def _extract_debt_plus_idle(success, AssetBreakdown):
        if success:
            totalIdle, totalDebt, totalDebtMin, totalDebtMax = AssetBreakdown
            return int(totalIdle + totalDebt) / 1e18
        return None

    calls = []
    for autopool in autopools:
        calls.extend(
            [
                Call(
                    autopool.autopool_vault_address,
                    ["totalSupply()(uint256)"],
                    [((autopool.autopool_vault_address, "total_shares"), safe_normalize_with_bool_success)],
                ),
                Call(
                    autopool.autopool_vault_address,
                    ["getAssetBreakdown()((uint256,uint256,uint256,uint256))"],
                    [((autopool.autopool_vault_address, "total_nav"), _extract_debt_plus_idle)],
                ),
                Call(
                    autopool.autopool_vault_address,
                    ["convertToAssets(uint256)(uint256)", int(1e18)],
                    [((autopool.autopool_vault_address, "nav_per_share"), safe_normalize_with_bool_success)],
                ),
            ]
        )

    autopool_state_df = get_raw_state_by_blocks(calls, missing_blocks, chain, include_block_number=True)
    return autopool_state_df


def _extract_autopool_state_rows(autopool_state_df: pd.DataFrame):

    new_autopool_states = []


def ensure_autopool_states_is_current():
    for chain in ALL_CHAINS:
        possible_blocks = build_blocks_to_use(chain)

        missing_blocks = get_subset_not_already_in_column(
            AutopoolStates,
            AutopoolStates.block,
            possible_blocks,
            where_clause=AutopoolStates.chain_id == chain.chain_id,
        )

        if len(missing_blocks) == 0:
            continue

        autopools = get_full_table_as_orm(Autopools, where_clause=Autopools.chain_id == chain.chain_id)

        autopool_state_df = _fetch_autopool_state_df(autopools, missing_blocks, chain)
        return autopool_state_df


autopool_state_df = ensure_autopool_states_is_current()
autopool_state_df

2025-04-27 20:07:38,507 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-27 20:07:38,513 INFO sqlalchemy.engine.Engine SELECT max(blocks.block) AS block 
FROM blocks 
WHERE blocks.chain_id = %(chain_id_1)s AND blocks.block >= %(block_1)s AND blocks.block <= %(block_2)s GROUP BY date_trunc(%(date_trunc_1)s, blocks.datetime) ORDER BY date_trunc(%(date_trunc_2)s, blocks.datetime)
2025-04-27 20:07:38,513 INFO sqlalchemy.engine.Engine [generated in 0.00053s] {'chain_id_1': 1, 'block_1': 20752910, 'block_2': 100000000, 'date_trunc_1': 'day', 'date_trunc_2': 'day'}
2025-04-27 20:07:38,751 INFO sqlalchemy.engine.Engine COMMIT
2025-04-27 20:07:38,841 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-27 20:07:38,842 INFO sqlalchemy.engine.Engine 
            SELECT *
            FROM autopool_states
            WHERE autopool_states.chain_id = 1
        
2025-04-27 20:07:38,842 INFO sqlalchemy.engine.Engine [generated in 0.00043s] {}
2025-04-27 20:07:39,031 INFO sqlalchemy.engine.Engi

,"(0x0A2b94F6871c1D7A32Fe58E1ab5e6deA2f114E56, total_shares)","(0x0A2b94F6871c1D7A32Fe58E1ab5e6deA2f114E56, total_nav)","(0x0A2b94F6871c1D7A32Fe58E1ab5e6deA2f114E56, nav_per_share)","(0x6dC3ce9C57b20131347FDc9089D740DAf6eB34c5, total_shares)","(0x6dC3ce9C57b20131347FDc9089D740DAf6eB34c5, total_nav)","(0x6dC3ce9C57b20131347FDc9089D740DAf6eB34c5, nav_per_share)","(0xE800e3760FC20aA98c5df6A9816147f190455AF3, total_shares)","(0xE800e3760FC20aA98c5df6A9816147f190455AF3, total_nav)","(0xE800e3760FC20aA98c5df6A9816147f190455AF3, nav_per_share)","(0x35911af1B570E26f668905595dEd133D01CD3E5a, total_shares)","(0x35911af1B570E26f668905595dEd133D01CD3E5a, total_nav)","(0x35911af1B570E26f668905595dEd133D01CD3E5a, nav_per_share)",block
timestamp,,,,,,,,,,,,,
2024-09-15 23:59:59+00:00,0.239949,0.239949,1.000000,1.000000e-13,1.000000e-13,1.000000,0.449884,0.449884,1.000000,NaN,NaN,NaN,20759464
2024-09-16 23:59:59+00:00,944.081116,944.081116,1.000000,6.109170e+02,6.107142e+02,0.999668,398.277626,398.277626,1.000000,NaN,NaN,NaN,20766617
2024-09-17 23:59:59+00:00,1654.392363,1654.384535,0.999995,1.025542e+03,1.024945e+03,0.999418,641.320724,641.244644,0.999881,NaN,NaN,NaN,20773761
2024-09-18 23:59:59+00:00,3133.620123,3133.327068,0.999906,1.817276e+03,1.815990e+03,0.999293,1149.219706,1148.786322,0.999623,NaN,NaN,NaN,20780916
2024-09-19 23:59:59+00:00,3903.814760,3901.645343,0.999444,1.947166e+03,1.945543e+03,0.999166,1389.852544,1389.059253,0.999429,NaN,NaN,NaN,20788084
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-04-23 23:59:59+00:00,13320.745844,13784.771713,1.034835,1.081396e+03,1.117050e+03,1.032970,478.869466,494.136743,1.031882,849.224032,860.944075,1.013801,22335153
2025-04-24 23:59:59+00:00,13340.566677,13806.582560,1.034932,1.078953e+03,1.114605e+03,1.033043,478.874032,494.267685,1.032146,849.244736,861.046513,1.013897,22342307
2025-04-25 23:59:59+00:00,14054.739402,14546.090669,1.034960,1.044624e+03,1.079424e+03,1.033313,478.871718,494.262104,1.032139,831.800530,843.439884,1.013993,22349480


In [3]:
wide_autopool_destination_df = natural_left_right_using_where(
    AutopoolDestinationStates,
    DestinationStates,
    using=[
        AutopoolDestinationStates.chain_id,
        AutopoolDestinationStates.block,
        AutopoolDestinationStates.destination_vault_address,
    ],
)

wide_autopool_destination_df

2025-04-27 20:06:40,302 INFO sqlalchemy.engine.Engine select pg_catalog.version()
2025-04-27 20:06:40,303 INFO sqlalchemy.engine.Engine [raw sql] {}
2025-04-27 20:06:40,490 INFO sqlalchemy.engine.Engine select current_schema()
2025-04-27 20:06:40,491 INFO sqlalchemy.engine.Engine [raw sql] {}
2025-04-27 20:06:40,677 INFO sqlalchemy.engine.Engine show standard_conforming_strings
2025-04-27 20:06:40,678 INFO sqlalchemy.engine.Engine [raw sql] {}
2025-04-27 20:06:40,866 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-27 20:06:40,867 INFO sqlalchemy.engine.Engine SELECT pg_catalog.pg_class.relname 
FROM pg_catalog.pg_class JOIN pg_catalog.pg_namespace ON pg_catalog.pg_namespace.oid = pg_catalog.pg_class.relnamespace 
WHERE pg_catalog.pg_class.relname = %(table_name)s AND pg_catalog.pg_class.relkind = ANY (ARRAY[%(param_1)s, %(param_2)s, %(param_3)s, %(param_4)s, %(param_5)s]) AND pg_catalog.pg_table_is_visible(pg_catalog.pg_class.oid) AND pg_catalog.pg_namespace.nspname != %(nspname

,chain_id,block,destination_vault_address,autopool_vault_address,amount,total_safe_value,total_spot_value,total_backing_value,percent_ownership,incentive_apr,...,underlying_token_total_supply,safe_total_supply,price_per_share,price_return,underlying_safe_price,underlying_spot_price,underlying_backing,safe_backing_discount,spot_backing_discount,safe_spot_spread
0,1,20759464,0xf3ae3c74EaD129e770A864CeE291A805b170bBe0,0x0A2b94F6871c1D7A32Fe58E1ab5e6deA2f114E56,0.000000,0.000000,0.000000,0.000000,0.0,0.041235,...,13157.451200,12302.611124,1.036523,-0.000645,1.036523,1.036516,1.035855,0.000645,0.000638,0.000007
1,1,20759464,0xf3ae3c74EaD129e770A864CeE291A805b170bBe0,0x6dC3ce9C57b20131347FDc9089D740DAf6eB34c5,0.000000,0.000000,0.000000,0.000000,0.0,0.041235,...,13157.451200,12302.611124,1.036523,-0.000645,1.036523,1.036516,1.035855,0.000645,0.000638,0.000007
2,1,20766617,0xf3ae3c74EaD129e770A864CeE291A805b170bBe0,0x0A2b94F6871c1D7A32Fe58E1ab5e6deA2f114E56,0.000000,0.000000,0.000000,0.000000,0.0,0.040938,...,13157.496413,12383.288934,1.036163,-0.000278,1.036163,1.036121,1.035875,0.000278,0.000237,0.000041
3,1,20766617,0xf3ae3c74EaD129e770A864CeE291A805b170bBe0,0x6dC3ce9C57b20131347FDc9089D740DAf6eB34c5,0.000000,0.000000,0.000000,0.000000,0.0,0.040938,...,13157.496413,12383.288934,1.036163,-0.000278,1.036163,1.036121,1.035875,0.000278,0.000237,0.000041
4,1,20773761,0xf3ae3c74EaD129e770A864CeE291A805b170bBe0,0x0A2b94F6871c1D7A32Fe58E1ab5e6deA2f114E56,0.000000,0.000000,0.000000,0.000000,0.0,0.040507,...,13157.496413,12383.288934,1.036462,-0.000538,1.036462,1.036456,1.035906,0.000537,0.000531,0.000006
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16855,8453,29331726,0xAADf01DD90aE0A6Bb9Eb908294658037096E0404,0xAADf01DD90aE0A6Bb9Eb908294658037096E0404,0.648453,0.648453,0.648453,0.648453,100.0,NaN,...,NaN,NaN,1.000000,0.000000,1.000000,1.000000,1.000000,0.000000,0.000000,0.000000
16856,8453,29374926,0xAADf01DD90aE0A6Bb9Eb908294658037096E0404,0xAADf01DD90aE0A6Bb9Eb908294658037096E0404,0.751798,0.751798,0.751798,0.751798,100.0,NaN,...,NaN,NaN,1.000000,0.000000,1.000000,1.000000,1.000000,0.000000,0.000000,0.000000
16857,8453,29418126,0xAADf01DD90aE0A6Bb9Eb908294658037096E0404,0xAADf01DD90aE0A6Bb9Eb908294658037096E0404,0.752076,0.752076,0.752076,0.752076,100.0,NaN,...,NaN,NaN,1.000000,0.000000,1.000000,1.000000,1.000000,0.000000,0.000000,0.000000
16858,8453,29461326,0xAADf01DD90aE0A6Bb9Eb908294658037096E0404,0xAADf01DD90aE0A6Bb9Eb908294658037096E0404,0.752076,0.752076,0.752076,0.752076,100.0,NaN,...,NaN,NaN,1.000000,0.000000,1.000000,1.000000,1.000000,0.000000,0.000000,0.000000


In [ ]:
down_weighted_apr_values = []


# this so that we can sum up the values
def _extract_down_weighted_apr_values(row: dict) -> None:
    autopool_total_nav = row[(row["autopool_vault_address"], "total_nav")]
    this_destination_safe_value = row["total_safe_value"]
    portion_of_value_in_destination = this_destination_safe_value / autopool_total_nav

    downscaled_total_apr_out = row.get("total_apr_out", 0) * portion_of_value_in_destination
    downscaled_total_apr_in = row.get("total_apr_in", 0) * portion_of_value_in_destination
    downscaled_safe_backing_discount = row.get("total_apr_in", 0) * portion_of_value_in_destination

    down_weighted_apr_values.append(
        {
            "downscaled_total_apr_out": downscaled_total_apr_out,
            "downscaled_total_apr_in": downscaled_total_apr_in,
            "downscaled_safe_backing_discount": downscaled_safe_backing_discount,
            "autopool_vault_address": row["autopool_vault_address"],
            "block": row["block"],
        }
    )


# not certain what this is
very_wide_df.apply(_extract_down_weighted_apr_values, axis=1)

down_weighted_apr_df = pd.DataFrame.from_records(down_weighted_apr_values)
weighted_average_df = down_weighted_apr_df.groupby(["autopool_vault_address", "block"]).sum()
weighted_average_df.columns = [
    "weighted_average_total_apr_out",
    "weighted_average_total_apr_in",
    "weighted_average_safe_backing_discount",
]
weighted_average_df = weighted_average_df.reset_index()

weighted_average_df

,autopool_vault_address,block,weighted_average_total_apr_out,weighted_average_total_apr_in,weighted_average_safe_backing_discount
0,0x0A2b94F6871c1D7A32Fe58E1ab5e6deA2f114E56,20759464,0.000000,0.000000,0.000000
1,0x0A2b94F6871c1D7A32Fe58E1ab5e6deA2f114E56,20766617,0.000000,0.000000,0.000000
2,0x0A2b94F6871c1D7A32Fe58E1ab5e6deA2f114E56,20773761,0.006059,0.006024,0.006024
3,0x0A2b94F6871c1D7A32Fe58E1ab5e6deA2f114E56,20780916,0.016019,0.015922,0.015922
4,0x0A2b94F6871c1D7A32Fe58E1ab5e6deA2f114E56,20788084,0.033280,0.033024,0.033024
...,...,...,...,...,...
895,0xE800e3760FC20aA98c5df6A9816147f190455AF3,22335153,0.084246,0.083994,0.083994
896,0xE800e3760FC20aA98c5df6A9816147f190455AF3,22342307,0.084246,0.083810,0.083810
897,0xE800e3760FC20aA98c5df6A9816147f190455AF3,22349480,0.084911,0.084568,0.084568
898,0xE800e3760FC20aA98c5df6A9816147f190455AF3,22356658,0.084474,0.084067,0.084067


Index([                                       'autopool_vault_address',
                                                               'block',
                                      'weighted_average_total_apr_out',
                                       'weighted_average_total_apr_in',
                              'weighted_average_safe_backing_discount',
        ('0x0A2b94F6871c1D7A32Fe58E1ab5e6deA2f114E56', 'total_shares'),
           ('0x0A2b94F6871c1D7A32Fe58E1ab5e6deA2f114E56', 'total_nav'),
       ('0x0A2b94F6871c1D7A32Fe58E1ab5e6deA2f114E56', 'nav_per_share'),
        ('0x6dC3ce9C57b20131347FDc9089D740DAf6eB34c5', 'total_shares'),
           ('0x6dC3ce9C57b20131347FDc9089D740DAf6eB34c5', 'total_nav'),
       ('0x6dC3ce9C57b20131347FDc9089D740DAf6eB34c5', 'nav_per_share'),
        ('0xE800e3760FC20aA98c5df6A9816147f190455AF3', 'total_shares'),
           ('0xE800e3760FC20aA98c5df6A9816147f190455AF3', 'total_nav'),
       ('0xE800e3760FC20aA98c5df6A9816147f190455AF3', 'nav_per_s

In [40]:
autopool_weighted_avgs_with_state_df.head(1).iloc[0]

autopool_vault_address                                         0x0A2b94F6871c1D7A32Fe58E1ab5e6deA2f114E56
block                                                                                            20759464
weighted_average_total_apr_out                                                                        0.0
weighted_average_total_apr_in                                                                         0.0
weighted_average_safe_backing_discount                                                                0.0
(0x0A2b94F6871c1D7A32Fe58E1ab5e6deA2f114E56, total_shares)                                       0.239949
(0x0A2b94F6871c1D7A32Fe58E1ab5e6deA2f114E56, total_nav)                                          0.239949
(0x0A2b94F6871c1D7A32Fe58E1ab5e6deA2f114E56, nav_per_share)                                           1.0
(0x6dC3ce9C57b20131347FDc9089D740DAf6eB34c5, total_shares)                                            0.0
(0x6dC3ce9C57b20131347FDc9089D740DAf6eB34c5, t

In [44]:
new_autopool_state_rows = []
chain = ALL_CHAINS[0]


def _extract_autopool_state(row: dict):
    new_autopool_state_rows.append(
        AutopoolStates(
            autopool_vault_address=row["autopool_vault_address"],
            block=int(row["block"]),
            chain_id=chain.chain_id,
            total_shares=row[(row["autopool_vault_address"], "total_shares")],
            total_nav=row[(row["autopool_vault_address"], "total_nav")],
            nav_per_share=row[(row["autopool_vault_address"], "nav_per_share")],
            weighted_average_total_apr_out=row["weighted_average_total_apr_out"],
            weighted_average_total_apr_in=row["weighted_average_total_apr_in"],
            weighted_average_safe_backing_discount=row["weighted_average_safe_backing_discount"],
        )
    )


autopool_weighted_avgs_with_state_df.apply(_extract_autopool_state, axis=1)

new_autopool_state_rows

[AutopoolStates(autopool_vault_address='0x0A2b94F6871c1D7A32Fe58E1ab5e6deA2f114E56', chain_id=1, block=20759464, total_shares=0.23994860869514073, total_nav=0.23994860869514073, nav_per_share=1.0, weighted_average_total_apr_out=0.0, weighted_average_total_apr_in=0.0, weighted_average_safe_backing_discount=0.0),
 AutopoolStates(autopool_vault_address='0x0A2b94F6871c1D7A32Fe58E1ab5e6deA2f114E56', chain_id=1, block=20766617, total_shares=944.0811157622477, total_nav=944.0811157622477, nav_per_share=1.0, weighted_average_total_apr_out=0.0, weighted_average_total_apr_in=0.0, weighted_average_safe_backing_discount=0.0),
 AutopoolStates(autopool_vault_address='0x0A2b94F6871c1D7A32Fe58E1ab5e6deA2f114E56', chain_id=1, block=20773761, total_shares=1654.3923628226087, total_nav=1654.3845347543415, nav_per_share=0.9999952683121349, weighted_average_total_apr_out=0.00605904752973822, weighted_average_total_apr_in=0.006023685909272614, weighted_average_safe_backing_discount=0.006023685909272614),
 A

In [ ]:
# full

In [27]:
autopool_state_df

,"(0x0A2b94F6871c1D7A32Fe58E1ab5e6deA2f114E56, total_shares)","(0x0A2b94F6871c1D7A32Fe58E1ab5e6deA2f114E56, total_nav)","(0x0A2b94F6871c1D7A32Fe58E1ab5e6deA2f114E56, nav_per_share)","(0x6dC3ce9C57b20131347FDc9089D740DAf6eB34c5, total_shares)","(0x6dC3ce9C57b20131347FDc9089D740DAf6eB34c5, total_nav)","(0x6dC3ce9C57b20131347FDc9089D740DAf6eB34c5, nav_per_share)","(0xE800e3760FC20aA98c5df6A9816147f190455AF3, total_shares)","(0xE800e3760FC20aA98c5df6A9816147f190455AF3, total_nav)","(0xE800e3760FC20aA98c5df6A9816147f190455AF3, nav_per_share)","(0x35911af1B570E26f668905595dEd133D01CD3E5a, total_shares)","(0x35911af1B570E26f668905595dEd133D01CD3E5a, total_nav)","(0x35911af1B570E26f668905595dEd133D01CD3E5a, nav_per_share)",block
timestamp,,,,,,,,,,,,,
2024-09-15 23:59:59+00:00,0.239949,0.239949,1.000000,1.000000e-13,1.000000e-13,1.000000,0.449884,0.449884,1.000000,NaN,NaN,NaN,20759464
2024-09-16 23:59:59+00:00,944.081116,944.081116,1.000000,6.109170e+02,6.107142e+02,0.999668,398.277626,398.277626,1.000000,NaN,NaN,NaN,20766617
2024-09-17 23:59:59+00:00,1654.392363,1654.384535,0.999995,1.025542e+03,1.024945e+03,0.999418,641.320724,641.244644,0.999881,NaN,NaN,NaN,20773761
2024-09-18 23:59:59+00:00,3133.620123,3133.327068,0.999906,1.817276e+03,1.815990e+03,0.999293,1149.219706,1148.786322,0.999623,NaN,NaN,NaN,20780916
2024-09-19 23:59:59+00:00,3903.814760,3901.645343,0.999444,1.947166e+03,1.945543e+03,0.999166,1389.852544,1389.059253,0.999429,NaN,NaN,NaN,20788084
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-04-23 23:59:59+00:00,13320.745844,13784.771713,1.034835,1.081396e+03,1.117050e+03,1.032970,478.869466,494.136743,1.031882,849.224032,860.944075,1.013801,22335153
2025-04-24 23:59:59+00:00,13340.566677,13806.582560,1.034932,1.078953e+03,1.114605e+03,1.033043,478.874032,494.267685,1.032146,849.244736,861.046513,1.013897,22342307
2025-04-25 23:59:59+00:00,14054.739402,14546.090669,1.034960,1.044624e+03,1.079424e+03,1.033313,478.871718,494.262104,1.032139,831.800530,843.439884,1.013993,22349480


In [ ]:
# weighted_average_total_apr_out: Mapped[float] = mapped_column(nullable=False)
# weighted_average_total_apr_in: Mapped[float] = mapped_column(nullable=False)
# weighted_average_safe_backing_discount: Mapped[float] = mapped_column(nullable=False)  # price return

In [5]:
wide_autopool_destination_df.columns

Index(['chain_id', 'block', 'destination_vault_address',
       'autopool_vault_address', 'amount', 'total_safe_value',
       'total_spot_value', 'total_backing_value', 'percent_ownership',
       'incentive_apr', 'fee_apr', 'base_apr', 'points_apr',
       'fee_plus_base_apr', 'total_apr_in', 'total_apr_out',
       'underlying_token_total_supply', 'safe_total_supply', 'price_per_share',
       'price_return', 'underlying_safe_price', 'underlying_spot_price',
       'underlying_backing', 'safe_backing_discount', 'spot_backing_discount',
       'safe_spot_spread'],
      dtype='object')

In [ ]:
autopool_total_destination_safe_value_by_block = (
    wide_autopool_destination_df.groupby(["autopool_vault_address", "block"])["total_safe_value"].sum().reset_index()
)
autopool_total_destination_safe_value_by_block.columns = [
    "autopool_vault_address",
    "block",
    "autopool_total_safe_value",
]

,autopool_vault_address,block,total_safe_value
0,0x0A2b94F6871c1D7A32Fe58E1ab5e6deA2f114E56,20759464,0.000000
1,0x0A2b94F6871c1D7A32Fe58E1ab5e6deA2f114E56,20766617,0.000000
2,0x0A2b94F6871c1D7A32Fe58E1ab5e6deA2f114E56,20773761,99.989238
3,0x0A2b94F6871c1D7A32Fe58E1ab5e6deA2f114E56,20780916,560.414095
4,0x0A2b94F6871c1D7A32Fe58E1ab5e6deA2f114E56,20788084,1429.848335
...,...,...,...
1067,0xE800e3760FC20aA98c5df6A9816147f190455AF3,22327989,491.964071
1068,0xE800e3760FC20aA98c5df6A9816147f190455AF3,22335153,492.076268
1069,0xE800e3760FC20aA98c5df6A9816147f190455AF3,22342307,491.988405
1070,0xE800e3760FC20aA98c5df6A9816147f190455AF3,22349480,492.079276


In [2]:
limited_destination_states_df = destination_states_df[
    [
        "destination_vault_address",
        "block",
        "underlying_token_total_supply",
        "underlying_safe_price",
        "underlying_spot_price",
        "underlying_backing",
        "chain_id",
    ]
].copy()
autopool_destination_balance_of_df

,block,autopool_vault_address,destination_vault_address,balance_of
0,20759464.0,0x0A2b94F6871c1D7A32Fe58E1ab5e6deA2f114E56,0xf3ae3c74EaD129e770A864CeE291A805b170bBe0,0.000000
1,20759464.0,0x0A2b94F6871c1D7A32Fe58E1ab5e6deA2f114E56,0x865e59D439BF7310c9BC6117E6020B8C87De4065,0.000000
2,20759464.0,0x0A2b94F6871c1D7A32Fe58E1ab5e6deA2f114E56,0x25cb41919d6B88e0D48108A4F5fe8FBb3744aFE1,0.000000
3,20759464.0,0x0A2b94F6871c1D7A32Fe58E1ab5e6deA2f114E56,0x6DcB6797b1C0442587c2ad79745ef7BB487Fc2E2,0.000000
4,20759464.0,0x0A2b94F6871c1D7A32Fe58E1ab5e6deA2f114E56,0xe3AE2Ab9AE8ADe1B4940dd893C9339401bEe61A1,0.000000
...,...,...,...,...
15003,22356658.0,0xE800e3760FC20aA98c5df6A9816147f190455AF3,0x60339056EC88996e41757E05a798310E46972cca,0.000000
15004,22356658.0,0xE800e3760FC20aA98c5df6A9816147f190455AF3,0x5c6aeb9ef0d5BbA4E6691f381003503FD0D45126,470.519208
15005,22356658.0,0x35911af1B570E26f668905595dEd133D01CD3E5a,0xc4Eb861e7b66f593482a3D7E8adc314f6eEDA30B,317.661117
15006,22356658.0,0x35911af1B570E26f668905595dEd133D01CD3E5a,0xe4433D00Cf48BFE0C672d9949F2cd2c008bffC04,0.000000
